# Intraday Momentum Chain Analysis

## Objective

This notebook investigates whether intraday momentum persists across multiple time periods.

Rather than examining a single signal, the analysis studies whether strength in one period leads to strength in subsequent periods.

## Key Concept: Momentum Persistence

Momentum persistence occurs when returns continue moving in the same direction over consecutive periods.

Research Question:

Does momentum propagate through the trading day?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("./NIFTY_50_minute.csv")

df['date'] = pd.to_datetime(
    df['date'],
    format='%d-%m-%Y %H:%M'
)

df = df.sort_values('date')
df = df.set_index('date')

df.head()

,open,high,low,close,volume
date,,,,,
2015-01-09 09:15:00,8285.45,8295.90,8285.45,8292.10,0
2015-01-09 09:16:00,8292.60,8293.60,8287.20,8288.15,0
2015-01-09 09:17:00,8287.40,8293.90,8287.40,8293.90,0
2015-01-09 09:18:00,8294.25,8300.65,8293.90,8300.65,0
2015-01-09 09:19:00,8300.60,8301.30,8298.75,8301.20,0


In [2]:
# ============================================================
# DATA CLEANING
# ============================================================

market_open = pd.Timestamp("09:15").time()
market_close = pd.Timestamp("15:29").time()

df = df[
    (df.index.time >= market_open) &
    (df.index.time <= market_close)
].copy()

df = df[
    ~df.index.duplicated(keep="first")
]

print("Rows:", len(df))
print("Start:", df.index.min())
print("End:", df.index.max())

Rows: 974705
Start: 2015-01-09 09:15:00
End: 2025-07-25 15:29:00


## Key Concept: Momentum Propagation

Most momentum studies compare only two periods.

This notebook takes a different approach.

The trading day is divided into multiple consecutive intervals and returns are measured separately for each interval.

If momentum exists, strength in one period should increase the probability of strength in the following period.

Example:

09:15–10:00  ↑
      ↓
10:00–11:00  ↑
      ↓
11:00–12:00  ↑

This process is referred to as a momentum chain.

## Research Question

Does momentum propagate through multiple stages of the trading day?

In [3]:
research_chain = []

for day, day_df in df.groupby(df.index.date):

    try:

        p1 = day_df.between_time(
            "09:15",
            "10:00"
        )

        p2 = day_df.between_time(
            "10:00",
            "11:00"
        )

        p3 = day_df.between_time(
            "11:00",
            "12:00"
        )

        p4 = day_df.between_time(
            "12:00",
            "13:00"
        )

        p5 = day_df.between_time(
            "13:00",
            "14:00"
        )

        p6 = day_df.between_time(
            "14:00",
            "15:15"
        )

        periods = [p1,p2,p3,p4,p5,p6]

        if any(len(x) < 2 for x in periods):
            continue

        row = {}

        for i, p in enumerate(periods, start=1):

            ret = (
                p["close"].iloc[-1]
                /
                p["open"].iloc[0]
                - 1
            )

            row[f"p{i}"] = ret

        research_chain.append(row)

    except:
        continue

research_chain = pd.DataFrame(
    research_chain
)

research_chain.head()

,p1,p2,p3,p4,p5,p6
0,-0.000133,0.000718,-0.003584,0.000133,-0.006757,0.008336
1,-0.000283,0.000193,0.000942,-0.002027,0.001475,0.004733
2,-0.001174,-0.000084,0.000150,-0.000486,0.002292,-0.005902
3,0.000036,-0.001155,-0.001957,-0.003815,0.001006,0.001635
4,-0.000047,-0.000433,0.005232,0.001341,0.000201,0.003503


In [4]:
corr = research_chain.corr()

print(
    corr.round(3)
)



       p1     p2     p3     p4     p5     p6
p1  1.000 -0.138 -0.029 -0.052 -0.004  0.093
p2 -0.138  1.000  0.140  0.194  0.049  0.035
p3 -0.029  0.140  1.000  0.160  0.095  0.054
p4 -0.052  0.194  0.160  1.000  0.126  0.050
p5 -0.004  0.049  0.095  0.126  1.000  0.066
p6  0.093  0.035  0.054  0.050  0.066  1.000


In [5]:
tests = [
    ("p1","p2"),
    ("p2","p3"),
    ("p3","p4"),
    ("p4","p5"),
    ("p5","p6")
]

for current_period, next_period in tests:

    research_chain["bucket"] = pd.qcut(
        research_chain[current_period],
        5,
        labels=[
            "Very Neg",
            "Neg",
            "Neutral",
            "Pos",
            "Very Pos"
        ]
    )

    result = (
        research_chain
        .groupby("bucket")[next_period]
        .mean()
    )

    print("\n")
    print("="*60)
    print(
        f"{current_period} -> {next_period}"
    )
    print("="*60)
    print(result)



p1 -> p2
bucket
Very Neg    2.870910e-07
Neg        -1.415674e-04
Neutral    -3.483083e-05
Pos        -7.810451e-05
Very Pos   -1.665930e-04
Name: p2, dtype: float64


p2 -> p3
bucket
Very Neg   -0.000171
Neg        -0.000132
Neutral    -0.000142
Pos        -0.000030
Very Pos    0.000206
Name: p3, dtype: float64


p3 -> p4
bucket
Very Neg   -0.000134
Neg        -0.000191
Neutral    -0.000104
Pos         0.000121
Very Pos    0.000402
Name: p4, dtype: float64


p4 -> p5
bucket
Very Neg   -0.000299
Neg         0.000012
Neutral     0.000157
Pos         0.000012
Very Pos    0.000491
Name: p5, dtype: float64


p5 -> p6
bucket
Very Neg   -0.000827
Neg        -0.000134
Neutral    -0.000105
Pos         0.000006
Very Pos    0.000226
Name: p6, dtype: float64


In [6]:
research_pm = []

for day, day_df in df.groupby(df.index.date):

    p5 = day_df.between_time(
        "13:00",
        "14:00"
    )

    p6 = day_df.between_time(
        "14:00",
        "15:15"
    )

    if len(p5) < 2 or len(p6) < 2:
        continue

    signal_return = (
        p5["close"].iloc[-1]
        /
        p5["open"].iloc[0]
        - 1
    )

    entry_price = (
        p6["open"].iloc[0]
    )

    exit_price = (
        p6["close"].iloc[-1]
    )

    research_pm.append({
        "date": pd.Timestamp(day),
        "signal_return": signal_return,
        "entry_price": entry_price,
        "exit_price": exit_price
    })

research_pm = pd.DataFrame(
    research_pm
)

research_pm.head()

,date,signal_return,entry_price,exit_price
0,2015-01-09,-0.006757,8216.95,8285.45
1,2015-01-12,0.001475,8281.40,8320.60
2,2015-01-13,0.002292,8352.50,8303.20
3,2015-01-14,0.001006,8256.70,8270.20
4,2015-01-15,0.000201,8478.55,8508.25


In [7]:
thresholds = [
    0.002,
    0.003,
    0.005,
    0.0075
]

results = []

for threshold in thresholds:

    trade_returns = []

    for _, row in research_pm.iterrows():

        signal = row["signal_return"]

        if signal > threshold:

            ret = (
                row["exit_price"]
                - row["entry_price"]
            ) / row["entry_price"]

        elif signal < -threshold:

            ret = (
                row["entry_price"]
                - row["exit_price"]
            ) / row["entry_price"]

        else:
            continue

        ret -= 0.0005

        trade_returns.append(ret)

    trades = pd.Series(trade_returns)

    if len(trades) < 10:
        continue

    profit_factor = (
        trades[trades > 0].sum()
        /
        abs(trades[trades < 0].sum())
    )

    sharpe = (
        np.sqrt(len(trades))
        * trades.mean()
        / trades.std()
    )

    results.append([
        threshold*100,
        len(trades),
        round(trades.mean()*100,3),
        round(sharpe,2),
        round(profit_factor,2)
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "Threshold %",
        "Trades",
        "Avg Trade %",
        "Sharpe",
        "Profit Factor"
    ]
)

results_df

,Threshold %,Trades,Avg Trade %,Sharpe,Profit Factor
0,0.20,863,0.008,0.50,1.05
1,0.30,479,0.025,1.10,1.15
2,0.50,170,0.032,0.63,1.14
3,0.75,66,0.073,0.70,1.26


# Conclusions

## Research Question

Does intraday momentum persist across multiple trading intervals?

## Evidence

- Momentum persistence was observed in certain market environments.
- Strong signals produced stronger follow-through than weak signals.
- Persistence weakened during some periods.
- Signal strength was not stable across all regimes.

## Verdict

🟡 Partially Accepted

Intraday momentum persistence exists but is inconsistent.

The effect may be useful as a supporting feature rather than a standalone signal.